# PSF Defocus Z-Scan Analysis

Vectorial Debye PSF simulation sweep for **ATTO 488**, **ATTO 565**, and **ATTO 647N**.  
Quantifies the effect of axial defocus on localisation precision and colour precision  
using the standard S3M fitting pipeline (`FittingStrategy.STANDARD`).

Physics model: vectorial Debye + Gibson–Lanni spherical aberration (NA 1.49, oil/water).

**Structure**
1. PSF shape visualisation — per-channel (B/G/R) RGB composite at increasing defocus  
2. Z-sweep simulation — 7 z-offsets × 2 coverslip depths × 3 photon levels  
3. Analysis — localisation precision and colour precision vs defocus

In [ ]:
import sys, os, glob, types
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import polars as pl

sys.path.append(str(Path(os.getcwd()).parent.parent))

from src import IOFunctions, SpectralFunctions, MaskFunctions, sCMOSFunctions, PlottingBase
from src.Multicolour_Simulation_Functions import (
    FittingStrategy, SimulationConfig, MultiC_Sim_Funcs,
)
from src.simulation.defocus_psf import VectorialPSF

IO    = IOFunctions.IO_Functions()
S_F   = SpectralFunctions.Spectral_Funcs()
M_F   = MaskFunctions.Mask_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
MSF   = MultiC_Sim_Funcs()
plotter = PlottingBase.PublicationPlotter()

In [ ]:
# ── Camera calibration ────────────────────────────────────────────────────────
cal_folder = Path("../../Camera_Calibrations/Ximea_Camera/")
gain      = IO.read_tiff(str(cal_folder / "gain.tif"))
offset    = IO.read_tiff(str(cal_folder / "offset.tif"))
variance  = IO.read_tiff(str(cal_folder / "variance.tif"))
readnoise = float(np.median(IO.read_tiff(str(cal_folder / "readnoise.tif"))))
rqe       = IO.read_tiff(str(cal_folder / "rqe.tif"))

print(f"Camera calibration loaded — sensor: {gain.shape}, readnoise: {readnoise:.2f}")

In [ ]:
# ── Spectral setup ────────────────────────────────────────────────────────────
R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])   # (3, n_wl); convention B=0, G=1, R=2

# Dyes and global simulation constants
dyes          = ["ATTO 488", "ATTO 565", "ATTO 647N"]
filters       = []          # no bandpass filter
NA            = 1.49
pixel_size_nm = 69.0        # Ximea object-space pixel (nm)
image_dims    = 22          # 22 × 22 px  ≈  1.52 µm FOV
background_photons = 40.0

# Smoothing function (Gaussian, σ = 1.5 px) ─ must be defined before MSF calls
smoothing_function = types.SimpleNamespace(
    args={"sigma": 1.5},
    extent=1.5,
    smoothing_function=sCMOS.gaussian_filter_stack,
    data_arg="image",
)

# Coarser wavelength grid for PSF integration (5 nm steps; sufficient to <1% PSF error)
wl_psf_nm = np.arange(400, 751, 5, dtype=float)

# build_spectral_weights requires pixel_QYs on the same grid as wl_psf_nm;
# the native grid from getpixelefficiency() may differ — interpolate.
pixel_QYs_psf = np.vstack([
    np.interp(wl_psf_nm, wavelength, pixel_QYs[c])
    for c in range(pixel_QYs.shape[0])
])  # (3, len(wl_psf_nm))

print(f"Wavelength grid: {wavelength[0]:.0f}–{wavelength[-1]:.0f} nm  "
      f"({len(wavelength)} pts);  PSF grid: {len(wl_psf_nm)} pts @ 5 nm steps")

In [ ]:
# ── Save folder ───────────────────────────────────────────────────────────────
save_folder = Path("/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20260623_DefocusZScan")
save_folder.mkdir(parents=True, exist_ok=True)
print(f"Results → {save_folder}")

In [ ]:
# ── VectorialPSF objects and per-dye spectral weights ─────────────────────────
# N_pupil=128 is fast enough for the display figure; simulation dispatch uses 256
vpsf_display = VectorialPSF(
    NA=NA, n_medium=1.33, n_immersion=1.515,
    pix_obj_um=pixel_size_nm / 1000.0,
    psf_size=21, N_pupil=128,
)

# Spectral weights: w_c(λ) = S(λ)·T_obj(λ)·QE_c(λ), shape (n_wl_psf, 3) = (n_wl_psf, B/G/R)
# pixel_QYs_psf is already interpolated to wl_psf_nm in the spectral setup cell.
spectral_weights = {}
channel_fractions = {}
for dye in dyes:
    w = VectorialPSF.build_spectral_weights(
        spectral_functions=S_F,
        dye=dye,
        filters=None,
        wavelengths_nm=wl_psf_nm,
        pixel_QYs=pixel_QYs_psf,   # (3, n_wl_psf) — matched grid
        include_objective=True,
    )  # (n_wl_psf, 3)
    spectral_weights[dye] = w
    frac = w.sum(axis=0) / w.sum()
    channel_fractions[dye] = frac
    print(f"{dye:12s}  B={frac[0]:.3f}  G={frac[1]:.3f}  R={frac[2]:.3f}")

## PSF Shape Visualisation

Polychromatic vectorial PSF at each defocus step, integrated over the dye emission  
spectrum weighted by per-channel Bayer QE.  Each panel is an **RGB composite**:  
B-channel PSF → blue, G-channel PSF → green, R-channel PSF → red.  
Intensities are normalised to the peak of the **in-focus** PSF for that channel,  
so dimming of the central peak with defocus is visible.

In [ ]:
# ── Compute PSF patches for visualisation ────────────────────────────────────
z_display_nm = np.array([0, 100, 200, 300, 500], dtype=float)
z_display_um = z_display_nm / 1000.0
wl_display_um = wl_psf_nm / 1000.0

psf_display = {}    # psf_display[dye] shape: (n_z, 3, psf_size, psf_size)
for dye in dyes:
    psf_display[dye] = vpsf_display.compute_psf_stack(
        z_offsets_um=z_display_um,
        wavelengths_um=wl_display_um,
        spectral_weights=spectral_weights[dye],
        distance_from_coverslip_um=0.0,
    )  # (n_z, 3, 21, 21)
    print(f"{dye} PSF stack: {psf_display[dye].shape}  "
          f"in-focus B/G/R peak: {psf_display[dye][0].max(axis=(-2,-1))}")

In [ ]:
# ── Figure 1: defocused PSF shapes as imaged by the RGB camera ───────────────
# Layout: 3 rows (dyes) × 5 columns (z-offsets), two-column width
n_z_disp = len(z_display_nm)
n_dyes   = len(dyes)

# Height: ~1.45" per row to keep panels approximately square
fig, axs = plotter.two_column_plot(nrows=n_dyes, ncols=n_z_disp, height=4.5)

fs = plotter.config.font_size

for i_dye, dye in enumerate(dyes):
    stack = psf_display[dye]   # (n_z, 3, 21, 21);  channels: B=0, G=1, R=2

    # Normalise relative to the in-focus peak of each channel independently.
    # Shape: (3, 1, 1) → broadcast over (n_z, ps, ps)
    peak_infocus = stack[0].max(axis=(-2, -1))[:, np.newaxis, np.newaxis]  # (3, 1, 1)
    peak_infocus = np.where(peak_infocus > 0, peak_infocus, 1.0)

    for i_z in range(n_z_disp):
        ax = axs[i_dye, i_z]

        # Normalise this z-plane to in-focus peak
        psf_z = stack[i_z] / peak_infocus   # (3, ps, ps), values ≤ 1

        # RGB composite: matplotlib expects (H, W, 3) with order [R, G, B]
        # PSF channel order is [B=0, G=1, R=2]
        rgb = np.stack([psf_z[2], psf_z[1], psf_z[0]], axis=-1)   # [R, G, B]
        rgb = np.clip(rgb, 0.0, 1.0)

        ax.imshow(rgb, origin="upper", interpolation="nearest", aspect="equal")
        ax.set_axis_off()

        # Column header on top row
        if i_dye == 0:
            ax.set_title(f"z = {z_display_nm[i_z]:.0f} nm", fontsize=fs, pad=2)

    # Dye row label on left margin
    axs[i_dye, 0].text(
        -0.15, 0.5, dye,
        transform=axs[i_dye, 0].transAxes,
        fontsize=fs, rotation=90, va="center", ha="right",
    )

# Pixel scale bar annotation on bottom-right panel
ax_br = axs[-1, -1]
ax_br.set_axis_off()
ps_px = vpsf_display.psf_size
scale_nm = 200
scale_px = scale_nm / pixel_size_nm
x0_bar = ps_px - 1 - scale_px
ax_br.plot([x0_bar, x0_bar + scale_px], [ps_px - 1.5, ps_px - 1.5],
           lw=1.5, color="white", solid_capstyle="butt")
ax_br.text(x0_bar + scale_px / 2, ps_px - 2.8, f"{scale_nm} nm",
           fontsize=fs - 1, color="white", ha="center", va="bottom")

fig.tight_layout(pad=0.2, h_pad=0.3, w_pad=0.2)
plotter.save_or_show(fig, save_path=str(save_folder / "psf_defocus_shapes.pdf"))

## Z-Sweep Simulation

For each (dye, z-offset, coverslip depth), run `test_simulation_method` with  
`FittingStrategy.STANDARD` across three photon levels.  
Two coverslip depths probe the effect of depth-induced spherical aberration:
- **0 nm** — emitter at the coverslip (no spherical aberration)  
- **500 nm** — emitter 500 nm into the aqueous sample (realistic cell imaging)

In [ ]:
# ── Sweep parameters ─────────────────────────────────────────────────────────
z_offsets_nm       = np.array([0, 50, 100, 150, 200, 300, 400, 500], dtype=float)
coverslip_depths_nm = np.array([0, 500], dtype=float)
n_photon_space     = np.array([500, 1000, 2000], dtype=float)

# n_bootstrap: number of simulated localisations per (z, photon, dye, d_coverslip).
# 2000 gives ~3% RMSE uncertainty (1/√2000); increase to 10 000 for publication quality.
n_bootstrap = 2000

print(f"Sweep:  {len(dyes)} dyes  ×  {len(z_offsets_nm)} z-offsets  "
      f"×  {len(coverslip_depths_nm)} coverslip depths  "
      f"×  {len(n_photon_space)} photon levels")
print(f"Total simulation calls: "
      f"{len(dyes) * len(z_offsets_nm) * len(coverslip_depths_nm)}")

In [ ]:
# ── Camera parameters dictionary (constant across sweep) ─────────────────────
def make_camera_params(image_dims):
    """Median-flat camera calibration for a square image of size image_dims."""
    return {
        "gain":               np.full((image_dims, image_dims), np.median(gain)),
        "offset":             np.full((image_dims, image_dims), np.median(offset)),
        "variance":           np.full((image_dims, image_dims), np.median(variance)),
        "readnoise":          readnoise,
        "rqe":                np.full((image_dims, image_dims), np.median(rqe)),
        "masks":              M_F.get_masks(size_x=image_dims, size_y=image_dims),
        "pixel_QYs":          pixel_QYs,
        "pixel_order":        ["B", "G", "R"],
        "pixel_order_indices": {"B": 0, "G": 1, "R": 2},
    }

camera_params_dict = make_camera_params(image_dims)
print(f"Camera parameters dict ready — image size: {image_dims}×{image_dims} px")

In [ ]:
# ── Run sweep ─────────────────────────────────────────────────────────────────
import time

n_total  = len(dyes) * len(z_offsets_nm) * len(coverslip_depths_nm)
i_run    = 0
t_start  = time.time()

for dye in dyes:
    for z_nm in z_offsets_nm:
        for d_nm in coverslip_depths_nm:
            i_run += 1
            flag = f"defocus_z{z_nm:04.0f}nm_d{d_nm:04.0f}nm_"

            config = SimulationConfig(
                n_bootstrap=n_bootstrap,
                background_photons=background_photons,
                NA=NA,
                pixel_size=pixel_size_nm,
                save_raw_results=True,
                subtractx0y0=False,    # keep absolute positions; residuals computed in analysis
                use_stochastic_photons=True,
                save_summary_csvs=True,
                verbose=False,
                defocus_z_um=z_nm / 1000.0,
                distance_from_coverslip_um=d_nm / 1000.0,
            )

            print(f"[{i_run:3d}/{n_total}]  {dye}  z={z_nm:4.0f} nm  "
                  f"d={d_nm:4.0f} nm", end="  ", flush=True)

            MSF.test_simulation_method(
                dye=dye,
                filters=filters,
                wavelength=wavelength,
                camera_parameters=camera_params_dict,
                save_folder=str(save_folder),
                n_photon_space=n_photon_space,
                smoothing_function=smoothing_function,
                strategy=FittingStrategy.STANDARD,
                starting_flag=flag,
                config=config,
                overwrite=True,
            )

            elapsed = (time.time() - t_start) / 60.0
            rate    = i_run / elapsed if elapsed > 0 else 0
            eta     = (n_total - i_run) / rate if rate > 0 else 0
            print(f"done   ({elapsed:.1f} min elapsed, ETA {eta:.1f} min)")

print(f"\nSweep complete in {(time.time()-t_start)/60:.1f} min.")

## Analysis

Load the RMSE summary CSVs saved by `test_simulation_method` and aggregate into  
an xarray for plotting.  

Metrics (from `_compute_fit_statistics`):
- **`xc`, `yc`** — RMSE of fitted position in nm  
- **`colour_distance`** — mean Euclidean distance of fitted (A_B, A_G, A_R) from truth

In [ ]:
# ── Load RMSE summary CSVs ────────────────────────────────────────────────────
results = {}   # key: (dye, z_nm, d_nm, n_photon) → dict of metrics
missing = []

for dye in dyes:
    dye_str = dye.replace("/", "-")
    for z_nm in z_offsets_nm:
        for d_nm in coverslip_depths_nm:
            flag = f"defocus_z{z_nm:04.0f}nm_d{d_nm:04.0f}nm_"
            pattern = str(save_folder / f"{flag}*{dye_str}*RMSE_mean*.csv")
            files = sorted(glob.glob(pattern))
            if not files:
                missing.append((dye, z_nm, d_nm))
                continue
            df = pl.read_csv(files[0])
            for row in df.iter_rows(named=True):
                n_ph = float(row["n_photons"])
                results[(dye, z_nm, d_nm, n_ph)] = {
                    "sigma_x":        row["xc"],
                    "sigma_y":        row["yc"],
                    "sigma_xy":       float(np.sqrt((row["xc"]**2 + row["yc"]**2) / 2)),
                    "colour_dist":    row["colour_distance"],
                }

print(f"Loaded {len(results)} result entries.")
if missing:
    print(f"Missing: {missing}")

In [ ]:
# ── Plotting helpers ──────────────────────────────────────────────────────────
# Dye colours matched to approximate emission
dye_colours = {"ATTO 488": "#1565C0", "ATTO 565": "#E65100", "ATTO 647N": "#B71C1C"}
dye_labels  = {"ATTO 488": "ATTO 488", "ATTO 565": "ATTO 565", "ATTO 647N": "ATTO 647N"}

# Line style: solid = at coverslip (d=0), dashed = 500 nm into sample
depth_ls   = {0.0: "-", 500.0: "--"}
depth_alpha= {0.0: 1.0, 500.0: 0.65}

fs = plotter.config.font_size

In [ ]:
# ── Figure 2: localisation precision σ_xy vs defocus ─────────────────────────
# One panel per photon level, two-column width
n_ph_levels = len(n_photon_space)
fig, axs = plotter.two_column_plot(nrows=1, ncols=n_ph_levels, height=2.5)

for i_ph, n_ph in enumerate(n_photon_space):
    ax = axs[0, i_ph]
    for dye in dyes:
        for d_nm in coverslip_depths_nm:
            y = []
            for z_nm in z_offsets_nm:
                key = (dye, z_nm, d_nm, float(n_ph))
                if key in results:
                    y.append(results[key]["sigma_xy"])
                else:
                    y.append(np.nan)
            label = f"{dye_labels[dye]} (d={d_nm:.0f} nm)" if i_ph == 0 else None
            ax.plot(
                z_offsets_nm, y,
                color=dye_colours[dye],
                ls=depth_ls[d_nm],
                alpha=depth_alpha[d_nm],
                lw=1.0,
                label=label,
            )
    ax.set_xlabel("Defocus / nm", fontsize=fs)
    ax.set_ylabel(r"$\sigma_{xy}$ / nm", fontsize=fs) if i_ph == 0 else None
    ax.set_title(f"{n_ph:.0f} photons", fontsize=fs)
    ax.tick_params(labelsize=fs - 1)
    ax.set_xlim([0, z_offsets_nm.max()])
    ax.set_ylim(bottom=0)

# Legend on first panel
axs[0, 0].legend(fontsize=fs - 1, frameon=False, loc="upper left")

fig.tight_layout(pad=0.4)
plotter.save_or_show(fig, save_path=str(save_folder / "localisation_precision_vs_defocus.pdf"))

In [ ]:
# ── Figure 3: colour precision vs defocus ─────────────────────────────────────
fig, axs = plotter.two_column_plot(nrows=1, ncols=n_ph_levels, height=2.5)

for i_ph, n_ph in enumerate(n_photon_space):
    ax = axs[0, i_ph]
    for dye in dyes:
        for d_nm in coverslip_depths_nm:
            y = []
            for z_nm in z_offsets_nm:
                key = (dye, z_nm, d_nm, float(n_ph))
                if key in results:
                    y.append(results[key]["colour_dist"])
                else:
                    y.append(np.nan)
            label = f"{dye_labels[dye]} (d={d_nm:.0f} nm)" if i_ph == 0 else None
            ax.plot(
                z_offsets_nm, y,
                color=dye_colours[dye],
                ls=depth_ls[d_nm],
                alpha=depth_alpha[d_nm],
                lw=1.0,
                label=label,
            )
    ax.set_xlabel("Defocus / nm", fontsize=fs)
    ax.set_ylabel(r"$\sigma_{\mathrm{colour}}$ (a.u.)", fontsize=fs) if i_ph == 0 else None
    ax.set_title(f"{n_ph:.0f} photons", fontsize=fs)
    ax.tick_params(labelsize=fs - 1)
    ax.set_xlim([0, z_offsets_nm.max()])
    ax.set_ylim(bottom=0)

axs[0, 0].legend(fontsize=fs - 1, frameon=False, loc="upper left")

fig.tight_layout(pad=0.4)
plotter.save_or_show(fig, save_path=str(save_folder / "colour_precision_vs_defocus.pdf"))

In [ ]:
# ── Figure 4: combined summary at 1000 photons ────────────────────────────────
# Two-column figure with localisation precision (left) and colour precision (right)
n_ph_summary = 1000.0

fig, axs = plotter.two_column_plot(nrows=1, ncols=2, height=2.5)
ax_loc, ax_col = axs[0, 0], axs[0, 1]

for dye in dyes:
    for d_nm in coverslip_depths_nm:
        xy_list, col_list = [], []
        for z_nm in z_offsets_nm:
            key = (dye, z_nm, d_nm, n_ph_summary)
            if key in results:
                xy_list.append(results[key]["sigma_xy"])
                col_list.append(results[key]["colour_dist"])
            else:
                xy_list.append(np.nan)
                col_list.append(np.nan)
        label = f"{dye_labels[dye]}, d={d_nm:.0f} nm"
        kw = dict(color=dye_colours[dye], ls=depth_ls[d_nm],
                  alpha=depth_alpha[d_nm], lw=1.0, label=label)
        ax_loc.plot(z_offsets_nm, xy_list, **kw)
        ax_col.plot(z_offsets_nm, col_list, **kw)

for ax, ylabel in [
    (ax_loc, r"$\sigma_{xy}$ / nm"),
    (ax_col, r"$\sigma_{\mathrm{colour}}$ (a.u.)"),
]:
    ax.set_xlabel("Defocus / nm", fontsize=fs)
    ax.set_ylabel(ylabel, fontsize=fs)
    ax.tick_params(labelsize=fs - 1)
    ax.set_xlim([0, z_offsets_nm.max()])
    ax.set_ylim(bottom=0)

ax_loc.set_title(f"Localisation precision ({n_ph_summary:.0f} photons)", fontsize=fs)
ax_col.set_title(f"Colour precision ({n_ph_summary:.0f} photons)", fontsize=fs)
ax_loc.legend(fontsize=fs - 1, frameon=False, loc="upper left",
              ncols=1, handlelength=1.2)

fig.tight_layout(pad=0.4)
plotter.save_or_show(
    fig,
    save_path=str(save_folder / f"summary_{n_ph_summary:.0f}ph_defocus.pdf"),
)